In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import r2_score, accuracy_score, recall_score,precision_score, confusion_matrix

import pandas as pd
import numpy as np

In [2]:
# df = pd.read_csv(r"data\claims_train_scaled.csv")
df = pd.read_csv(r"../../data/claims_train_scaled.csv")
# df.drop("Area",axis=1,inplace=True)
# fuel_map = {'Regular': 0, 'Diesel': 1}
# df['VehGas'] = df['VehGas'].map(fuel_map)
# x = df.drop(columns=["ClaimNb","Region","VehBrand","VehGas"])
x = df.drop(columns=["ClaimNb"])
x = pd.get_dummies(x, columns=['Region','VehGas','VehBrand'], drop_first=True, dtype=float)

y = (df["ClaimNb"] > 0).astype(int)

x_train, x_val, y_train, y_val = train_test_split(x,y,test_size=0.2,random_state=42)

In [7]:
# Retrieve the best parameters found by RandomizedSearchCV
best_params = random_search_clf.best_params_
print("Best Parameters:", best_params)

# Evaluate the model with best parameters on the test set
best_estimator = random_search_clf.best_estimator_
val_accuracy = best_estimator.score(x_val, y_val)
print("Test r2 with Best Parameters:", val_accuracy)

Best Parameters: {'rf__oob_score': True, 'rf__n_estimators': 200, 'rf__min_samples_split': 4, 'rf__min_samples_leaf': 1, 'rf__max_features': 'log2', 'rf__max_depth': 20, 'rf__bootstrap': True, 'mlp__hidden_layer_sizes': (192, 48, 192), 'mlp__batch_size': 1000, 'mlp__alpha': 0.0001, 'lsvc__tol': 1e-06, 'lsvc__penalty': 'l2', 'lsvc__loss': 'hinge', 'lsvc__C': 0.75, 'gbr__max_features': 0.75, 'gbr__max_depth': 20, 'gbr__learning_rate': 0.2}
Test r2 with Best Parameters: 0.9492255369487265


In [8]:
y_val_pred = best_estimator.predict(x_val)
confusion_matrix(y_val,y_val_pred)

array([[102628,    136],
       [  5358,     82]])

In [9]:
val_accuracy

0.9492255369487265

In [10]:
precision_score(y_val,y_val_pred)

0.3761467889908257

In [11]:
recall_score(y_val,y_val_pred)

0.015073529411764706

In [6]:
from sklearn.model_selection import RandomizedSearchCV 

estimators = [
    ('rf', RandomForestClassifier(random_state=42, criterion='log_loss')),
    ('lsvc', LinearSVC(random_state=42)),
    ('mlp', MLPClassifier(random_state=42, max_iter=100, learning_rate_init=0.001, activation='relu', verbose=True, early_stopping=True, tol=1e-6)),
    ('gbr', HistGradientBoostingClassifier(
                max_iter=1000,
                verbose=0,
                # categorical_features=["Region","VehBrand","VehGas"],
            )),
]

clf = StackingClassifier(
    estimators=estimators, final_estimator=LogisticRegression(), verbose=1, n_jobs=-1
)

params = {    
    'rf__n_estimators': [200],
    'rf__max_depth': [5,10,15,20],
    'rf__min_samples_leaf': [1,2,4],
    'rf__min_samples_split': [2,3,4],
    'rf__max_features': ['sqrt', 'log2', None],
    'rf__bootstrap': [True, False],
    'rf__oob_score': [True, False],
    'lsvc__penalty': ['l1','l2'],
    'lsvc__loss': ['hinge','squared_hinge'],
    'lsvc__tol': [1e-4,1e-6,1e-8],
    'lsvc__C': [0.5,0.75,1,1.25],
    'mlp__hidden_layer_sizes': [(192,),
                           (192,192),
                           (192,48),
                           (192,192,192),
                           (192,48,192),
                           (192,48,48)],
    'mlp__alpha': [0.0005,0.0001,0.00005],
    'mlp__batch_size': [100,1000,10000],
    'gbr__max_depth': [10,20,None],
    'gbr__learning_rate' : [0.05,0.1,0.2],
    'gbr__max_features': [0.5,0.75,1.0],
}

random_search_clf = RandomizedSearchCV(clf, params, n_jobs=-1, verbose=1, scoring="recall", cv=5, n_iter=50, random_state=42)

random_search_clf.fit(x_train,y_train)
# clf.fit(x_train, y_train).score(x_val, y_val)

Binning 0.076 GB of training data: 1.540 s
Binning 0.008 GB of validation data: 0.050 s
Fitting gradient boosted rounds:
Iteration 1, loss = 0.23772511
Validation score: 0.949856
Iteration 2, loss = 0.19880968
Validation score: 0.949856
Iteration 3, loss = 0.19516229
Validation score: 0.949856
Iteration 4, loss = 0.19307445
Validation score: 0.949856
Iteration 5, loss = 0.19184723
Validation score: 0.949856
Iteration 6, loss = 0.19096246
Validation score: 0.949856
Fit 76 trees in 16.090 s, (2356 total leaves)
Time spent computing histograms: 10.734s
Time spent finding best splits:  0.272s
Time spent applying splits:      1.184s
Time spent predicting:           0.080s
Iteration 7, loss = 0.19030574
Validation score: 0.949856
Binning 0.076 GB of training data: 1.139 s
Binning 0.008 GB of validation data: 0.093 s
Fitting gradient boosted rounds:
Iteration 8, loss = 0.18976478
Validation score: 0.949856
Iteration 9, loss = 0.18942997
Validation score: 0.949856
Iteration 10, loss = 0.189030

[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   40.8s finished


Iteration 4, loss = 0.19301539
Validation score: 0.949856
Iteration 5, loss = 0.19163122
Validation score: 0.949856
Iteration 5, loss = 0.19178287
Validation score: 0.949856
Fit 60 trees in 17.978 s, (1860 total leaves)
Time spent computing histograms: 12.241s
Time spent finding best splits:  0.332s
Time spent applying splits:      1.312s
Time spent predicting:           0.101s
Iteration 6, loss = 0.19075835
Validation score: 0.949856
Binning 0.076 GB of training data: Iteration 6, loss = 0.19091180
Validation score: 0.949856
1.417 s
Binning 0.008 GB of validation data: Iteration 7, loss = 0.19011257
0.093 s
Fitting gradient boosted rounds:
Validation score: 0.949856
Iteration 7, loss = 0.19026488
Validation score: 0.949856
Iteration 8, loss = 0.18949404
Validation score: 0.949856
Iteration 8, loss = 0.18971369
Validation score: 0.949856
Iteration 9, loss = 0.18916582
Validation score: 0.949856
Iteration 9, loss = 0.18929244
Validation score: 0.949819
Fit 77 trees in 18.539 s, (2387 to

[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   50.0s finished


Iteration 17, loss = 0.18680159
Validation score: 0.949783
Iteration 18, loss = 0.18671032
Validation score: 0.949928
Iteration 18, loss = 0.18662642
Validation score: 0.949819
Iteration 19, loss = 0.18641756
Validation score: 0.949892
Fit 98 trees in 27.220 s, (3038 total leaves)
Time spent computing histograms: 20.341s
Time spent finding best splits:  0.331s
Time spent applying splits:      1.826s
Time spent predicting:           0.119s
Binning 0.076 GB of training data: Iteration 19, loss = 0.18629465
Validation score: 0.949892
Binning 0.076 GB of training data: Fit 76 trees in 24.589 s, (2356 total leaves)
Time spent computing histograms: 18.056s
Time spent finding best splits:  0.309s
Time spent applying splits:      1.503s
Time spent predicting:           0.133s
2.939 s
Binning 0.008 GB of validation data: Iteration 20, loss = 0.18627733
Validation score: 0.949819
0.538 s
Fitting gradient boosted rounds:
Iteration 20, loss = 0.18613800
3.351 s
Binning 0.008 GB of validation data:

...

Iteration 11, loss = 0.18397624
Iteration 11, loss = 0.18427069
Validation score: 0.949892
Iteration 11, loss = 0.18412176
Iteration 11, loss = 0.18402288
Validation score: 0.949776
Validation score: 0.949863
Validation score: 0.949892
Iteration 11, loss = 0.18381854
Validation score: 0.949921
Iteration 12, loss = 0.18361797
Iteration 12, loss = 0.18389344
Validation score: 0.949776
Validation score: 0.949892
Iteration 12, loss = 0.18396340
Iteration 12, loss = 0.18352494
Validation score: 0.949921
Validation score: 0.949863
Iteration 12, loss = 0.18337439
Validation score: 0.949949
Iteration 13, loss = 0.18335141
Iteration 13, loss = 0.18360755
Validation score: 0.949949
Validation score: 0.949863
Iteration 13, loss = 0.18339388
Iteration 13, loss = 0.18309645
Validation score: 0.949632
Validation score: 0.949892
Iteration 13, loss = 0.18293230
Validation score: 0.949834
Iteration 14, loss = 0.18294780
Validation score: 0.949892
Iteration 14, loss = 0.18315826
Iteration 14, loss = 0.1

[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  2.4min remaining:  3.6min


Iteration 19, loss = 0.18045160
Validation score: 0.949718
Iteration 20, loss = 0.18045553
Validation score: 0.949718
Iteration 20, loss = 0.18057030
Validation score: 0.950007
Iteration 20, loss = 0.18025855
Validation score: 0.949574
Iteration 21, loss = 0.18002842
Validation score: 0.949747
Iteration 21, loss = 0.17973677
Validation score: 0.949892
Iteration 21, loss = 0.17960986
Validation score: 0.949921
Iteration 22, loss = 0.17963169
Validation score: 0.949603
Iteration 22, loss = 0.17944108
Validation score: 0.949892
Iteration 22, loss = 0.17932817
Validation score: 0.949949
Iteration 23, loss = 0.17945086
Validation score: 0.949487
Iteration 23, loss = 0.17892057
Validation score: 0.949718
Iteration 23, loss = 0.17877411
Validation score: 0.949805
Iteration 24, loss = 0.17900995
Validation score: 0.949718
Validation score did not improve more than tol=0.000001 for 10 consecutive epochs. Stopping.
Iteration 24, loss = 0.17849005
Validation score: 0.949834
Iteration 24, loss = 0

[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.9min finished


,estimator,"StackingClass...-1, verbose=2)"
,param_distributions,"{'gbr__learning_rate': [0.05, 0.1, ...], 'gbr__max_depth': [10, 20, ...], 'gbr__max_features': [0.5, 0.75, ...], 'lsvc__C': [0.5, 0.75, ...], ...}"
,n_iter,50
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan
